# Phase 3 - Layer C: Visibility Window Generation

This notebook combines the steallites from Layer A with the gateway locations from Layer B to generate physically grounded satellite-to-ground visibility windows.

A visibility window is a time interval when a satellite is visible from a ground station. In order to be so, the satellite must place itself above a minimum elevation angle as seen from a ground station. The elevation mask is configurable and represents the minimum useful angle above the local horizon. This first implementation uses sampled propagation, so the window boundaries are approximate to the configured time-step resolution.

## Configuration

The notebook reads the reviewed Layer A and Layer B CSV outputs, propagates a configurable subset over a given horizon, and writes the generated visibility windows to `data/dataset/starlink/intermediate`.

In [6]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

REQUIRED_PACKAGES = ["pandas", "skyfield"]
missing_packages = [
    package_name
    for package_name in REQUIRED_PACKAGES
    if importlib.util.find_spec(package_name) is None
]

if missing_packages:
    print(f"Installing missing packages into this notebook kernel: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Dependency installation finished. If imports still fail, restart the kernel and run the notebook again.")

import numpy as np
import pandas as pd
from skyfield.api import EarthSatellite, load, wgs84

# --------- loading configuration environment ----------
# load .json config file shared by the data wrangling notebooks
CONFIG_PATH = Path("data_config.json")
data_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

HORIZON_START_UTC = data_config["visibility_horizon_start_utc"]
HORIZON_HOURS = data_config["visibility_horizon_hours"]
TIME_STEP_SECONDS = data_config["visibility_time_step_seconds"]     # The elevation mask is the minimum angle above the 
                                                                    # local horizon at which you decide a satellite is 
                                                                    # "visible enough" to count as a usable contact.
ELEVATION_MASK_DEG = data_config["visibility_elevation_mask_deg"]
SATELLITE_SUBSET_SIZE = data_config["visibility_satellite_subset_size"]
GATEWAY_SUBSET_SIZE = data_config["visibility_gateway_subset_size"]
MIN_WINDOW_DURATION_SECONDS = data_config["visibility_min_window_duration_seconds"]

# --------- project directory setup ----------
# intermediate datasets: only consumed by the next data wrangling notebooks
INTERMEDIATE_DIR = Path("../../data/dataset/starlink/intermediate")

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
SATELLITE_INPUT_CSV = INTERMEDIATE_DIR / "starlink_satellite_subset.csv"
GATEWAY_INPUT_CSV = INTERMEDIATE_DIR / "starlink_gateways_clean.csv"
OUTPUT_CSV_PATH = INTERMEDIATE_DIR / "starlink_visibility_windows.csv"

print(f"Notebook Python executable: {sys.executable}")
print(f"Config path: {CONFIG_PATH.resolve()}")
print(f"Layer C intermediate data directory: {INTERMEDIATE_DIR.resolve()}")
print(f"Satellite input: {SATELLITE_INPUT_CSV.resolve()}")
print(f"Gateway input: {GATEWAY_INPUT_CSV.resolve()}")
print(f"Output CSV path: {OUTPUT_CSV_PATH.resolve()}")

Notebook Python executable: d:\University\PoliMi\courses\SWE2\SWE2-research-project\.venv\Scripts\python.exe
Config path: D:\University\PoliMi\courses\SWE2\SWE2-research-project\results\01_data_wrangling\data_config.json
Layer C intermediate data directory: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate
Satellite input: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate\starlink_satellite_subset.csv
Gateway input: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate\starlink_gateways_clean.csv
Output CSV path: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate\starlink_visibility_windows.csv


## Load Reviewed Layer A and Layer B Outputs

Layer C intentionally depends only on the CSV artifacts generated by the reviewed earlier layers. The subset sizes are configurable so the first review run stays fast and readable.

In [7]:
def load_layer_inputs(
    satellite_csv_path: Path,
    gateway_csv_path: Path,
    satellite_subset_size: int,
    gateway_subset_size: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load and trim the reviewed Layer A satellite and Layer B gateway tables.

    Parameters
    ----------
    satellite_csv_path:
        Path to the cleaned Starlink satellite CSV from Layer A.
    gateway_csv_path:
        Path to the cleaned gateway CSV from Layer B.
    satellite_subset_size:
        Maximum number of satellites to keep for this propagation run.
    gateway_subset_size:
        Maximum number of gateways to keep for this propagation run.

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame]
        Deterministically sorted satellite and gateway subsets.
    """

    # chec for invalid paths and wrong attributes
    if not satellite_csv_path.exists():
        raise FileNotFoundError(f"Missing Layer A satellite CSV: {satellite_csv_path}")
    if not gateway_csv_path.exists():
        raise FileNotFoundError(f"Missing Layer B gateway CSV: {gateway_csv_path}")
    if satellite_subset_size <= 0 or gateway_subset_size <= 0:
        raise ValueError("Subset sizes must be positive integers.")

    satellites_df = pd.read_csv(satellite_csv_path)
    gateways_df = pd.read_csv(gateway_csv_path)

    satellites_df = (
        satellites_df.sort_values(["norad_id", "satellite_name"])
        .head(satellite_subset_size)
        .reset_index(drop=True)
    )
    gateways_df = (
        gateways_df.sort_values(["station_id", "station_name"])
        .head(gateway_subset_size)
        .reset_index(drop=True)
    )

    return satellites_df, gateways_df


satellites, gateways = load_layer_inputs(
    SATELLITE_INPUT_CSV,
    GATEWAY_INPUT_CSV,
    SATELLITE_SUBSET_SIZE,
    GATEWAY_SUBSET_SIZE,
)

print(f"Satellites loaded for Layer C: {len(satellites)}")
print(f"Gateways loaded for Layer C: {len(gateways)}")
print("\nSatellite preview:")
print(satellites[["satellite_name", "norad_id", "epoch"]].head().to_string(index=False))
print("\nGateway preview:")
print(gateways[["station_id", "station_name", "latitude", "longitude", "status"]].head().to_string(index=False))

Satellites loaded for Layer C: 1000
Gateways loaded for Layer C: 150

Satellite preview:
satellite_name  norad_id                epoch
 STARLINK-1008     44714 2026-06-18T02:01:59Z
 STARLINK-1012     44718 2026-06-18T06:34:49Z
 STARLINK-1017     44723 2026-06-17T23:42:40Z
 STARLINK-1020     44725 2026-06-18T01:38:19Z
 STARLINK-1036     44741 2026-06-17T23:27:31Z

Gateway preview:
  station_id               station_name  latitude  longitude      status
GATEWAY_0001               Adelanto, CA   34.5828  -117.4090     planned
GATEWAY_0002            Aerzen, Germany   52.0489     9.2683 operational
GATEWAY_0003               Akita, Japan   39.7200   140.1024 operational
GATEWAY_0004                  Akron, OH   41.0575   -81.5199 operational
GATEWAY_0005 Alfouvar de Cima, Portugal   40.5167    -8.2500 operational


## Build the Propagation Timeline

We build the timegrid for satellite position in order to compute the visibility windows. The timegrid discretizes the whole horizon in steps of length `step_length`, and at each step the following code computes the sattelittes elevation with respect to a ground station.

In [8]:
def build_time_grid(start_utc: str, horizon_hours: int, step_seconds: int) -> tuple[pd.DatetimeIndex, object]:
    """Create pandas and Skyfield time grids for the propagation horizon.

    Parameters
    ----------
    start_utc:
        ISO-like UTC start time for the scheduling horizon.
    horizon_hours:
        Length of the horizon in hours.
    step_seconds:
        Sampling interval in seconds.

    Returns
    -------
    tuple[pandas.DatetimeIndex, skyfield.timelib.Time]
        Matching pandas timestamps and Skyfield time objects.
    """
    if horizon_hours <= 0:
        raise ValueError("horizon_hours must be positive.")
    if step_seconds <= 0:
        raise ValueError("step_seconds must be positive.")

    start = pd.Timestamp(start_utc).tz_convert("UTC")
    end = start + pd.Timedelta(hours=horizon_hours)
    timestamps = pd.date_range(start=start, end=end, freq=f"{step_seconds}s", inclusive="both")

    timescale = load.timescale()
    # due Skyfield requires to work with it's own time object we need to convert it
    skyfield_times = timescale.from_datetimes(timestamps.to_pydatetime())

    return timestamps, skyfield_times


time_grid, skyfield_times = build_time_grid(HORIZON_START_UTC, HORIZON_HOURS, TIME_STEP_SECONDS)

print(f"Horizon start: {time_grid[0].isoformat()}")
print(f"Horizon end: {time_grid[-1].isoformat()}")
print(f"Time samples: {len(time_grid)}")
print(f"Step seconds: {TIME_STEP_SECONDS}")
print(f"Elevation mask: {ELEVATION_MASK_DEG} degrees")

Horizon start: 2026-06-18T00:00:00+00:00
Horizon end: 2026-06-18T04:00:00+00:00
Time samples: 241
Step seconds: 60
Elevation mask: 25.0 degrees


## Compute Visibility Windows

For each satellite-gateway pair, the notebook computes satellite elevation over the horizon and groups contiguous samples above the elevation mask. The resulting start and end times are approximate to the configured sample step.

In [9]:
def extract_windows_from_elevation(
    visible_mask: np.ndarray,
    elevations_deg: np.ndarray,
    timestamps: pd.DatetimeIndex,
    min_duration_seconds: int,
) -> list[dict]:
    """Convert sampled visibility booleans into contiguous window records.

    Parameters
    ----------
    visible_mask:
        Boolean array where True means the satellite is above the elevation mask.
    elevations_deg:
        Elevation values in degrees for the same timestamps.
    timestamps:
        UTC timestamps corresponding to each sampled elevation.
    min_duration_seconds:
        Minimum approximate duration retained in the output.

    Returns
    -------
    list[dict]
        Window timing records with start, end, duration, and max elevation.
    """
    if len(visible_mask) != len(timestamps):
        raise ValueError("visible_mask and timestamps must have the same length.")
    
    # add False element to be sure that the visiibility windows has an end
    # even if starts/ends at the beginning/end of samples
    # [ F T T T F F T T ] --becomes--> [ F | F T T T F F T T | F ]
    padded = np.concatenate(([False], visible_mask, [False]))

    # convert the boolean visibility window in changing timestols:
    # -  0 if the visibility status of the satellite doesn't change
    # - +1 if the satelliste becomes visible
    # - -1 if the satellithe becomes not-visible
    changes = np.diff(padded.astype(int))

    # extract starting and ending visibility window indices
    start_indices = np.flatnonzero(changes == 1)
    end_indices = np.flatnonzero(changes == -1)
    
    # start building actual visibility windows for the satellites (expressed as 
    # timestamps and not as indices)
    windows = []
    for start_index, end_index in zip(start_indices, end_indices):
        # convert indicies in actual timestamps
        start_time = timestamps[start_index]
        if end_index < len(timestamps):
            end_time = timestamps[end_index]
        else:
            end_time = timestamps[-1]

        duration_seconds = int((end_time - start_time).total_seconds())
        if duration_seconds < min_duration_seconds:
            continue

        segment_elevations = elevations_deg[start_index:end_index]
        windows.append(
            {
                "start_time": start_time.isoformat(),
                "end_time": end_time.isoformat(),
                "duration_seconds": duration_seconds,
                "max_elevation_deg": round(float(np.nanmax(segment_elevations)), 3),
            }
        )

    return windows


def generate_visibility_windows(
    satellites_df: pd.DataFrame,
    gateways_df: pd.DataFrame,
    timestamps: pd.DatetimeIndex,
    skyfield_time_grid: object,
    elevation_mask_deg: float,
    min_duration_seconds: int,
) -> pd.DataFrame:
    """Generate visibility windows for every selected satellite-gateway pair.

    Parameters
    ----------
    satellites_df:
        Layer A satellite table with TLE lines and NORAD IDs.
    gateways_df:
        Layer B gateway table with station IDs and coordinates.
    timestamps:
        UTC timestamp labels for the propagation samples.
    skyfield_time_grid:
        Skyfield time object matching `timestamps`.
    elevation_mask_deg:
        Minimum elevation angle required for visibility.
    min_duration_seconds:
        Minimum approximate window duration retained in the output.

    Returns
    -------
    pandas.DataFrame
        Visibility-window table ready for downstream scheduling enrichment.
    """
    timescale = load.timescale()
    records = []

    gateway_objects = []
    for gateway in gateways_df.itertuples(index=False):
        gateway_objects.append(
            {
                "station_id": gateway.station_id,
                "station_name": gateway.station_name,
                "observer": wgs84.latlon(gateway.latitude, gateway.longitude),
            }
        )

    for satellite in satellites_df.itertuples(index=False):
        satellite_object = EarthSatellite(
            satellite.tle_line_1,
            satellite.tle_line_2,
            satellite.satellite_name,
            timescale,
        )

        for gateway in gateway_objects:
            #get the distance between staellite and gateway ...
            topocentric = (satellite_object - gateway["observer"]).at(skyfield_time_grid)
            altitude, _, _ = topocentric.altaz()
            # ... and extract the elevation degree to compute the visibility window
            elevations_deg = altitude.degrees
            visible_mask = elevations_deg >= elevation_mask_deg

            pair_windows = extract_windows_from_elevation(
                visible_mask,
                elevations_deg,
                timestamps,
                min_duration_seconds,
            )

            for window in pair_windows:
                records.append(
                    {
                        "satellite_name": satellite.satellite_name,
                        "norad_id": int(satellite.norad_id),
                        "station_id": gateway["station_id"],
                        "station_name": gateway["station_name"],
                        "start_time": window["start_time"],
                        "end_time": window["end_time"],
                        "duration_seconds": window["duration_seconds"],
                        "max_elevation_deg": window["max_elevation_deg"],
                        "source_epoch": satellite.epoch,
                    }
                )

    # just put the results into a dataframe for better usability
    windows_df = pd.DataFrame.from_records(records)
    windows_df = windows_df.sort_values(["start_time", "station_id", "norad_id"]).reset_index(drop=True)
    windows_df.insert(0, "window_id", [f"VIS_{index:06d}" for index in range(1, len(windows_df) + 1)])

    return windows_df


visibility_windows = generate_visibility_windows(
    satellites,
    gateways,
    time_grid,
    skyfield_times,
    ELEVATION_MASK_DEG,
    MIN_WINDOW_DURATION_SECONDS,
)

print(f"Generated visibility windows: {len(visibility_windows)}")
visibility_windows.head(10)

Generated visibility windows: 73600


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch
0,VIS_000001,STARLINK-1552,46076,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,2026-06-18T00:04:00+00:00,240,48.621,2026-06-17T21:51:22Z
1,VIS_000002,STARLINK-2495,48325,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,2026-06-18T00:03:00+00:00,180,50.028,2026-06-17T21:50:51Z
2,VIS_000003,STARLINK-2738,48668,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,2026-06-18T00:04:00+00:00,240,66.572,2026-06-17T21:52:22Z
3,VIS_000004,STARLINK-3159,49415,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,2026-06-18T00:02:00+00:00,120,32.655,2026-06-18T06:11:24Z
4,VIS_000005,STARLINK-3117,49435,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,2026-06-18T00:02:00+00:00,120,50.149,2026-06-18T06:08:48Z
5,VIS_000006,STARLINK-2356,47896,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,2026-06-18T00:03:00+00:00,180,79.033,2026-06-17T23:34:49Z
6,VIS_000007,STARLINK-3297,50159,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,2026-06-18T00:05:00+00:00,300,88.099,2026-06-17T07:41:29Z
7,VIS_000008,STARLINK-3289,50173,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,2026-06-18T00:02:00+00:00,120,38.942,2026-06-18T02:40:02Z
8,VIS_000009,STARLINK-3283,50179,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,2026-06-18T00:01:00+00:00,60,28.556,2026-06-18T02:40:46Z
9,VIS_000010,STARLINK-3267,50197,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,2026-06-18T00:03:00+00:00,180,44.213,2026-06-18T02:42:05Z


## Save Phase 3 Output

The generated visibility windows are saved as a CSV for later operational-constraint enrichment. This is still raw geometric visibility, not a final scheduling instance.

In [10]:
visibility_windows.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Saved visibility windows to: {OUTPUT_CSV_PATH.resolve()}")
print(f"Rows saved: {len(visibility_windows)}")
print(f"Satellites: {visibility_windows[["satellite_name"]].nunique()}")
print(f"Ground stations: {visibility_windows[["station_id"]].nunique()}")
visibility_windows.head(10)

Saved visibility windows to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink\intermediate\starlink_visibility_windows.csv
Rows saved: 73600
Satellites: satellite_name    983
dtype: int64
Ground stations: station_id    150
dtype: int64


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch
0,VIS_000001,STARLINK-1552,46076,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,2026-06-18T00:04:00+00:00,240,48.621,2026-06-17T21:51:22Z
1,VIS_000002,STARLINK-2495,48325,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,2026-06-18T00:03:00+00:00,180,50.028,2026-06-17T21:50:51Z
2,VIS_000003,STARLINK-2738,48668,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,2026-06-18T00:04:00+00:00,240,66.572,2026-06-17T21:52:22Z
3,VIS_000004,STARLINK-3159,49415,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,2026-06-18T00:02:00+00:00,120,32.655,2026-06-18T06:11:24Z
4,VIS_000005,STARLINK-3117,49435,GATEWAY_0001,"Adelanto, CA",2026-06-18T00:00:00+00:00,2026-06-18T00:02:00+00:00,120,50.149,2026-06-18T06:08:48Z
5,VIS_000006,STARLINK-2356,47896,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,2026-06-18T00:03:00+00:00,180,79.033,2026-06-17T23:34:49Z
6,VIS_000007,STARLINK-3297,50159,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,2026-06-18T00:05:00+00:00,300,88.099,2026-06-17T07:41:29Z
7,VIS_000008,STARLINK-3289,50173,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,2026-06-18T00:02:00+00:00,120,38.942,2026-06-18T02:40:02Z
8,VIS_000009,STARLINK-3283,50179,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,2026-06-18T00:01:00+00:00,60,28.556,2026-06-18T02:40:46Z
9,VIS_000010,STARLINK-3267,50197,GATEWAY_0002,"Aerzen, Germany",2026-06-18T00:00:00+00:00,2026-06-18T00:03:00+00:00,180,44.213,2026-06-18T02:42:05Z
